# Disease Prediction from Medical Data
## Phase 1 — Dataset Loading and Overview
Let's load the dataset and perform an initial analysis to understand the data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('../data/dataset.csv')

### First 5 rows of the dataset

In [ ]:
df.head()

### Last 5 rows of the dataset

In [ ]:
df.tail()

### Dataset Shape

In [ ]:
print(f'The dataset has {df.shape[0]} rows and {df.shape[1]} columns.')

### Column Names

In [ ]:
print(df.columns.tolist())

### Data Types

In [ ]:
df.dtypes

### Statistical Summary

In [ ]:
df.describe()

### Numerical and Categorical Features
**Numerical Features:** age, trestbps, chol, thalach, oldpeak, ca

**Categorical Features:** sex, cp, fbs, restecg, exang, slope, thal

**Target Variable:** target (0 = No Disease, 1 = Disease)

### Feature Description:
- **age**: age in years
- **sex**: sex (1 = male; 0 = female)
- **cp**: chest pain type (1: typical angina, 2: atypical angina, 3: non-anginal pain, 4: asymptomatic)
- **trestbps**: resting blood pressure (in mm Hg)
- **chol**: serum cholestoral in mg/dl
- **fbs**: fasting blood sugar > 120 mg/dl (1 = true; 0 = false)
- **restecg**: resting electrocardiographic results
- **thalach**: maximum heart rate achieved
- **exang**: exercise induced angina (1 = yes; 0 = no)
- **oldpeak**: ST depression induced by exercise relative to rest
- **slope**: the slope of the peak exercise ST segment
- **ca**: number of major vessels (0-3) colored by flourosopy
- **thal**: 3 = normal; 6 = fixed defect; 7 = reversable defect
- **target**: Diagnosis of heart disease (1 = disease present, 0 = disease not present)

## Phase 2 — Data Cleaning
We will check for missing values, duplicates, and outliers.

### Missing Value Detection

In [ ]:
df.isnull().sum()

The UCI dataset originally uses '?' for missing values which we converted to NaN during download. We can see missing values in `ca` and `thal`. Since they are few, we will impute them with the median or mode. Why? Dropping them reduces our small dataset, while imputation with mode preserves the distribution for categorical/ordinal data.

In [ ]:
df['ca'] = df['ca'].fillna(df['ca'].mode()[0])
df['thal'] = df['thal'].fillna(df['thal'].mode()[0])
print('Missing values after treatment:')
print(df.isnull().sum())

### Duplicate Detection and Removal

In [ ]:
print(f'Number of duplicate rows: {df.duplicated().sum()}')
df.drop_duplicates(inplace=True)
print(f'Dataset shape after removing duplicates: {df.shape}')

### Invalid Value Detection

In [ ]:
for col in df.columns:
    print(f"{col}: {df[col].unique()}")

All values are within the expected categorical ranges or numerical boundaries.

### Outlier Analysis
Outliers in medical data might represent genuine medical conditions (e.g., extremely high cholesterol). We will visualize them but we should be careful before removing them. If cholesterol is 500, it is a high but physiologically possible value (Hypercholesterolemia). Removing such points blindly could reduce model sensitivity to extreme disease cases.

## Phase 3 — Exploratory Data Analysis (EDA)
Let's explore relationships between features and the target variable.

### Target Class Distribution

In [ ]:
sns.countplot(x='target', data=df)
plt.title('Target Class Distribution (0 = No Disease, 1 = Disease)')
plt.show()

**Observation:** The classes are relatively balanced, which means we do not need to apply aggressive techniques like SMOTE. It tells us our baseline accuracy is around 50-54%.

### Age Distribution

In [ ]:
sns.histplot(df['age'], kde=True)
plt.title('Age Distribution')
plt.show()

**Observation:** Most patients are between 50 and 60 years old. The distribution is slightly left-skewed (older patients are more common in this dataset).

### Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.show()

**Observation:** `cp` (chest pain), `thalach` (max heart rate), and `slope` show positive correlation with the target. `exang` (exercise angina), `oldpeak`, `ca`, and `thal` show negative correlation. There are no highly collinear features (correlation > 0.9) that require removal.

### Boxplots for Numerical Features

In [ ]:
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
plt.figure(figsize=(15, 10))
for i, col in enumerate(num_cols):
    plt.subplot(2, 3, i+1)
    sns.boxplot(y=df[col])
    plt.title(f'Boxplot of {col}')
plt.tight_layout()
plt.show()

**Observation:** We can see outliers in `trestbps`, `chol`, and `oldpeak`. As discussed, these could be real medical states. For instance, a resting blood pressure of 180 is high but completely medically possible (Hypertension crisis). We will keep them.

### Disease vs Important Medical Features

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(x='target', y='thalach', data=df)
plt.title('Max Heart Rate vs Target')
plt.show()

**Observation:** Patients with heart disease tend to have a lower maximum heart rate achieved compared to those without.

## Phase 4 — Feature Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Feature/Target Separation
X = df.drop('target', axis=1)
y = df['target']

# Train-Test Split (use stratify to maintain class distribution)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scaling
scaler = StandardScaler()
# Fit only on training data to prevent data leakage
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set size: {X_train.shape[0]}')
print(f'Test set size: {X_test.shape[0]}')

## Phase 5 & 6 — Model Training & Comparison

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

results = []

for name, model in models.items():
    # Use scaled data for LR and SVM, unscaled for tree-based models
    if name in ["Logistic Regression", "SVM"]:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
        
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    })
    
    print(f"--- {name} ---")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print("\n")


### Model Comparison DataFrame

In [ ]:
results_df = pd.DataFrame(results)
display(results_df)

In [ ]:
results_df.set_index("Model")[["Accuracy", "Recall", "ROC-AUC"]].plot(kind='bar', figsize=(10, 6))
plt.title("Model Comparison")
plt.ylabel("Score")
plt.ylim(0, 1.1)
plt.xticks(rotation=0)
plt.show()

**Trade-offs in Medical Context:**
In this medical classification problem, **Recall (Sensitivity)** is more critical than Precision or Accuracy. A false negative (failing to predict the disease when the patient has it) can have fatal consequences, whereas a false positive (predicting the disease when the patient is healthy) only leads to further testing. We should prioritize the model with the highest Recall and robust ROC-AUC over pure Accuracy.

## Phase 7 — Hyperparameter Tuning
Let's tune the Random Forest and Logistic Regression. We use cross-validation (cv=5) only on the training set to prevent leakage.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Logistic Regression Tuning
lr_params = {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l2']}
lr_grid = GridSearchCV(LogisticRegression(random_state=42), lr_params, cv=5, scoring='recall')
lr_grid.fit(X_train_scaled, y_train)
print(f"Best LR Params: {lr_grid.best_params_}")
print(f"Tuned LR Recall: {recall_score(y_test, lr_grid.best_estimator_.predict(X_test_scaled)):.4f}")

# Random Forest Tuning
rf_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10]
}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_params, cv=5, scoring='recall')
rf_grid.fit(X_train, y_train)
print(f"Best RF Params: {rf_grid.best_params_}")
print(f"Tuned RF Recall: {recall_score(y_test, rf_grid.best_estimator_.predict(X_test)):.4f}")

## Phase 8 — Final Model
We select the best model based on Recall and overall stability. Logistic regression usually acts as a strong, interpretable baseline for this dataset.

In [ ]:
import joblib
import os

os.makedirs('../models', exist_ok=True)

final_model = lr_grid.best_estimator_
joblib.dump(final_model, '../models/final_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

print("Final model and Scaler saved successfully.")